In [1]:
import pandas as pd
from collections import Counter
import os

# 原始数据路径
raw_path = r"E:\ecommerce-user-growth-analysis\data\raw\UserBehavior.csv"

# 输出结果路径
output_path = r"E:\ecommerce-user-growth-analysis\data\processed\full_core_metrics.csv"

# 数据没有表头，所以手动指定列名
col_names = ["user_id", "item_id", "category_id", "behavior_type", "timestamp"]

# 每次读取 100 万行，避免一次性读取 3.67GB 导致电脑卡死
chunksize = 1000000

# 初始化统计变量
total_behaviors = 0

user_set = set()
item_set = set()
category_set = set()
buyer_set = set()

behavior_counter = Counter()

min_time = None
max_time = None

# 分块读取全量数据
for i, chunk in enumerate(pd.read_csv(raw_path, names=col_names, chunksize=chunksize)):
    # 累计总行为数
    total_behaviors += len(chunk)
    
    # 累计独立用户、商品、类目
    user_set.update(chunk["user_id"].dropna().unique())
    item_set.update(chunk["item_id"].dropna().unique())
    category_set.update(chunk["category_id"].dropna().unique())
    
    # 统计各行为类型数量
    behavior_counter.update(chunk["behavior_type"].dropna())
    
    # 统计购买用户
    buy_users = chunk.loc[chunk["behavior_type"] == "buy", "user_id"].dropna().unique()
    buyer_set.update(buy_users)
    
    # 转换时间戳，统计时间范围
    chunk["datetime"] = pd.to_datetime(chunk["timestamp"], unit="s", errors="coerce")
    current_min = chunk["datetime"].min()
    current_max = chunk["datetime"].max()
    
    if min_time is None or current_min < min_time:
        min_time = current_min
        
    if max_time is None or current_max > max_time:
        max_time = current_max
    
    print(f"已处理第 {i + 1} 个 chunk，累计处理 {total_behaviors:,} 行")

# 汇总核心指标
unique_users = len(user_set)
unique_items = len(item_set)
unique_categories = len(category_set)

pv_count = behavior_counter.get("pv", 0)
fav_count = behavior_counter.get("fav", 0)
cart_count = behavior_counter.get("cart", 0)
buy_count = behavior_counter.get("buy", 0)

buyer_count = len(buyer_set)
user_purchase_conversion_rate = buyer_count / unique_users if unique_users > 0 else 0

metrics = pd.DataFrame({
    "指标": [
        "总行为数",
        "独立用户数",
        "独立商品数",
        "独立类目数",
        "pv 浏览量",
        "fav 收藏数",
        "cart 加购数",
        "buy 购买数",
        "购买用户数",
        "用户购买转化率",
        "最早行为时间",
        "最晚行为时间"
    ],
    "数值": [
        total_behaviors,
        unique_users,
        unique_items,
        unique_categories,
        pv_count,
        fav_count,
        cart_count,
        buy_count,
        buyer_count,
        user_purchase_conversion_rate,
        min_time,
        max_time
    ]
})

metrics

已处理第 1 个 chunk，累计处理 1,000,000 行
已处理第 2 个 chunk，累计处理 2,000,000 行
已处理第 3 个 chunk，累计处理 3,000,000 行
已处理第 4 个 chunk，累计处理 4,000,000 行
已处理第 5 个 chunk，累计处理 5,000,000 行
已处理第 6 个 chunk，累计处理 6,000,000 行
已处理第 7 个 chunk，累计处理 7,000,000 行
已处理第 8 个 chunk，累计处理 8,000,000 行
已处理第 9 个 chunk，累计处理 9,000,000 行
已处理第 10 个 chunk，累计处理 10,000,000 行
已处理第 11 个 chunk，累计处理 11,000,000 行
已处理第 12 个 chunk，累计处理 12,000,000 行
已处理第 13 个 chunk，累计处理 13,000,000 行
已处理第 14 个 chunk，累计处理 14,000,000 行
已处理第 15 个 chunk，累计处理 15,000,000 行
已处理第 16 个 chunk，累计处理 16,000,000 行
已处理第 17 个 chunk，累计处理 17,000,000 行
已处理第 18 个 chunk，累计处理 18,000,000 行
已处理第 19 个 chunk，累计处理 19,000,000 行
已处理第 20 个 chunk，累计处理 20,000,000 行
已处理第 21 个 chunk，累计处理 21,000,000 行
已处理第 22 个 chunk，累计处理 22,000,000 行
已处理第 23 个 chunk，累计处理 23,000,000 行
已处理第 24 个 chunk，累计处理 24,000,000 行
已处理第 25 个 chunk，累计处理 25,000,000 行
已处理第 26 个 chunk，累计处理 26,000,000 行
已处理第 27 个 chunk，累计处理 27,000,000 行
已处理第 28 个 chunk，累计处理 28,000,000 行
已处理第 29 个 chunk，累计处理 29,000,000 行
已处理第 30 个 chunk，累计处理 30,000,000 

,指标,数值
0,总行为数,100150807
1,独立用户数,987994
2,独立商品数,4162024
3,独立类目数,9439
4,pv 浏览量,89716264
5,fav 收藏数,2888258
6,cart 加购数,5530446
7,buy 购买数,2015839
8,购买用户数,672404
9,用户购买转化率,0.680575


In [2]:
# 保存全量核心指标结果
os.makedirs(os.path.dirname(output_path), exist_ok=True)

metrics.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"全量核心指标已保存到：{output_path}")

全量核心指标已保存到：E:\ecommerce-user-growth-analysis\data\processed\full_core_metrics.csv
